<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson09/01_sktime.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 09 · Time series forecasting with sktime

In 2017, Google ran a [Kaggle competition](https://www.kaggle.com/competitions/web-traffic-time-series-forecasting): forecast how many people will visit 145,063 Wikipedia pages every day for the next two months. We will use the real data and try many forecasting methods, from "repeat last week" to transformer models trained on billions of time points.

All models come through one package, [sktime](https://www.sktime.net). Its idea is the same as scikit-learn's: every model has `fit` and `predict`, so swapping a model is a one-line change.

Plan:
1. The data
2. sktime in three lines
3. Naive models (and why they are hard to beat)
4. A metric: sMAPE
5. Decomposition: trend + weekly + yearly pattern
6. Classical statistical models: ETS, ARIMA, TBATS, Prophet
7. Machine learning: a time series is a table
8. Foundation models: TimesFM, Chronos-2, TiRex-2, Timestral
9. An ensemble
10. All pages at once
11. Backtesting: is one test window enough?

---
## 0. Setup

sktime keeps its core small. Models from other packages ("soft dependencies") need an extra install. Prophet, LightGBM and `transformers` are already in Colab.

*A CPU runtime is enough. A GPU makes section 8 faster.*

In [ ]:
!pip install -q "sktime==1.3.0" statsforecast pmdarima "chronos-forecasting>=2.0" tirex-2 git+https://github.com/mistralai/timestral.git

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # the packages below are chatty
import logging
logging.getLogger("cmdstanpy").disabled = True   # Prophet's log messages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sktime.utils.plotting import plot_series

---
## 1. The data

The original data is huge: 145,063 pages × 803 days. Most pages are almost never visited: half of them have fewer than about 100 visits a day, and 8,280 pages have a median of zero. Forecasting zeros is not much fun, so we kept **352 popular pages** (median above 10,000 visits a day) and added the Halloween and Oktoberfest pages, because their peaks fall inside our test window.

*(We also dropped special pages such as `Special:Search`, and adult sites. For 119 pages, the last training day was a 0 caused by missing data, so we replaced it with the value from one week earlier.)*

In [ ]:
URL = "https://raw.githubusercontent.com/simecek/dspracticum2026/main/lesson09/"

train = pd.read_csv(URL + "wiki_train.csv.gz", index_col="date", parse_dates=True).asfreq("D")
test = pd.read_csv(URL + "wiki_test.csv.gz", index_col="date", parse_dates=True).asfreq("D")

print("train:", train.shape, train.index.min().date(), "→", train.index.max().date())
print("test: ", test.shape, test.index.min().date(), "→", test.index.max().date())
train.iloc[-5:, :4]

**Look inside.** Rows are days, columns are pages. `.asfreq("D")` tells pandas that the rows are daily (sktime needs to know the frequency).

Notice the **gap**: train ends on Sep 10, test starts on Sep 13. In the competition, these two days were left for sending in the forecasts. So we forecast 3 to 64 days ahead.

A column name has four parts: `article_project_access_agent`. We kept only `all-access` (desktop + mobile) and `all-agents` (people + bots).

In [ ]:
train.columns.tolist()

**Optional: fewer pages.** Sections 2–9 use one page, so the number of pages does not matter there. Section 10 fits every model to every page: with all 352 pages, it takes about 12 minutes on a fast laptop, and longer on a Colab CPU. The cell below keeps only the busiest pages (median above 20,000 visits a day) plus Halloween and Oktoberfest, 44 pages in total, and section 10 takes a minute or two.

**If your computer is fast enough (or you use a GPU runtime), you do not have to do this:** set `FEWER_PAGES = False`, or skip the cell.

In [ ]:
FEWER_PAGES = True

if FEWER_PAGES:
    keep = (train.median() >= 20_000) | train.columns.str.startswith(("Halloween", "Oktoberfest"))
    train, test = train.loc[:, keep], test.loc[:, keep]

print(train.shape[1], "pages")

The pages differ a lot in size: the English main page has millions of visits a day, Halloween only a few thousand outside October.

In [ ]:
train.median().sort_values(ascending=False).astype(int)

We take the logarithm, `log(1 + visits)`. Two reasons:
- big and small pages become comparable,
- patterns are often **multiplicative** (Saturday has 20 % fewer visits than Monday, whether the page is big or small). The logarithm makes them **additive**, and most models assume additive patterns.

We forecast in log space and transform back with `np.expm1` when we evaluate.

In [ ]:
train_log = np.log1p(train)
test_log = np.log1p(test)

page = "Halloween_en.wikipedia.org_all-access_all-agents"
# page = np.random.choice(train.columns)   # Try it: a random page

y_train = train_log[page]
y_test = test_log[page]

plot_series(y_train, y_test, labels=["train", "test"], title=page);

A `pd.Series` with a daily `DatetimeIndex`: this is all sktime needs. You can see a yearly peak (31 October), a weekly wiggle, and spikes.

**Try it:** plot `train[page]` without the log. What happens to the weekly pattern?

---
## 2. sktime in three lines

Every sktime model ("forecaster") works the same way:

```python
forecaster = SomeForecaster(...)   # 1. choose a model and its settings
forecaster.fit(y_train)            # 2. learn from the past
y_pred = forecaster.predict(fh)    # 3. forecast the future
```

`fh` is the **forecasting horizon**: which future points we want. We can give dates directly:

In [ ]:
from sktime.forecasting.base import ForecastingHorizon

fh = ForecastingHorizon(y_test.index, is_relative=False)
fh

**Look inside.** The same horizon in "steps after the last training day" (the *cutoff*):

In [ ]:
fh.to_relative(cutoff=y_train.index[-1])

Steps 3 to 64, as expected (the 2-day gap). `np.arange(3, 65)` would give the same horizon.

Two small helpers: `smape` (explained in section 4) and `show`, which plots the last year of training data, the test data and any forecasts, and stores the score in a leaderboard.

In [ ]:
# @title Helpers: smape, show, leaderboard
def smape(y_true, y_pred):
    """Symmetric mean absolute percentage error, on the original scale (visits)."""
    a, f = np.expm1(np.asarray(y_true)), np.expm1(np.asarray(y_pred))
    return 100 * np.mean(2 * np.abs(f - a) / (np.abs(a) + np.abs(f)))

leaderboard = {}

def show(**preds):
    """show(name=y_pred, ...): plot the forecasts and add them to the leaderboard."""
    for name, y_pred in preds.items():
        leaderboard[name] = smape(y_test, y_pred)
        print(f"{name:25s} sMAPE = {leaderboard[name]:6.1f}")
    plot_series(y_train["2016-09":], y_test, *preds.values(),
                labels=["train", "test", *preds.keys()])

---
## 3. Naive models

Before anything clever, try something stupid. These **baselines** tell us what "good" means:

- `mean`: the average of the whole history,
- `last`: the last value, forever,
- `drift`: a straight line from the first to the last value.

In [ ]:
from sktime.forecasting.naive import NaiveForecaster

preds = {}
for strategy in ["mean", "last", "drift"]:
    forecaster = NaiveForecaster(strategy=strategy)
    forecaster.fit(y_train)
    preds[strategy] = forecaster.predict(fh)

show(**preds)

**Seasonal naive.** With `sp` ("seasonal period"), `last` repeats the last *season* instead of the last value:
- `sp=7`: repeat the last week,
- `sp=364`: repeat the same days of last year. Why 364 and not 365? 364 = 52 × 7, so a Monday is matched with a Monday.

In [ ]:
last_week = NaiveForecaster(strategy="last", sp=7).fit(y_train).predict(fh)
last_year = NaiveForecaster(strategy="last", sp=364).fit(y_train).predict(fh)

show(last_week=last_week, last_year=last_year)

For Halloween, only "last year" knows that a peak is coming. For a page without a yearly event, last year may be too old, and last week may be closer. Which one wins depends on the page. In the competition, simple medians of the last few weeks were famously hard to beat.

**Try it:** `NaiveForecaster(strategy="mean", sp=7, window_length=56)` is the average of the last 8 Mondays, 8 Tuesdays, … Is it better than `last_week`?

---
## 4. A metric: sMAPE

How wrong is a forecast? The competition used the **symmetric mean absolute percentage error**:

$$\text{sMAPE} = \frac{100}{n} \sum_{t} \frac{|F_t - A_t|}{(|A_t| + |F_t|)/2}$$

where $A_t$ is the actual value and $F_t$ the forecast. It is a percentage error, so a page with 1,000 visits and a page with 1,000,000 visits count the same. It goes from 0 (perfect) to 200 (forecast 0 when the truth is not, or vice versa).

sktime has it too:

In [ ]:
from sktime.performance_metrics.forecasting import MeanAbsolutePercentageError

sk_smape = MeanAbsolutePercentageError(symmetric=True)
print("ours:  ", smape(y_test, last_week))
print("sktime:", 100 * sk_smape(np.expm1(y_test), np.expm1(last_week)))

*(sktime returns a fraction, we multiply by 100. Note that we compare visits, not logarithms. The sMAPE of logarithms would look much better than it really is.)*

**Try it:** what is the sMAPE of a forecast that is always exactly 2× the truth? And 0.5× the truth? (Is the "symmetric" error really symmetric?)

---
## 5. Decomposition: trend + weekly + yearly

A classical trick: split the series into parts and forecast each part separately.

$$y_t = \text{trend}_t + \text{weekly}_t + \text{yearly}_t + \text{remainder}_t$$

**MSTL** (multiple seasonal-trend decomposition using LOESS) does the splitting:

In [ ]:
from statsmodels.tsa.seasonal import MSTL

parts = MSTL(y_train, periods=[7, 365]).fit()
parts.plot();

**Look inside.** Panel by panel: the data, the slow trend, the weekly pattern (a repeating 7-day wave), the yearly pattern (with the Halloween peak), and what is left (the residual: spikes nobody can predict).

To forecast: repeat the seasonal patterns, extend the trend with a simple model, and add them up. sktime wraps the fast implementation from the `statsforecast` package:

In [ ]:
from sktime.forecasting.statsforecast import StatsForecastMSTL

mstl = StatsForecastMSTL(season_length=[7, 365])
mstl.fit(y_train)
show(MSTL=mstl.predict(fh))

---
## 6. Classical statistical models

These models were invented between the 1950s and 2010s, and they are still used everywhere, because they are fast, need little data and are easy to explain.

**Exponential smoothing (ETS).** The forecast is a weighted average of the past, where the weights shrink exponentially: yesterday counts most, last month a little. "ETS" stands for Error, Trend, Seasonality: each part can be additive, multiplicative or missing. `AutoETS(auto=True)` tries the combinations and keeps the best one.

In [ ]:
from sktime.forecasting.ets import AutoETS

ets = AutoETS(auto=True, sp=7)
ets.fit(y_train)
print(ets.summary().tables[0])   # look inside: which model was chosen

In [ ]:
show(ETS=ets.predict(fh))

**ARIMA.** The value today is a linear combination of the values in the last few days (**A**uto**R**egressive part) and of the recent errors (**M**oving **A**verage part), after differencing the series to remove the trend (the **I**, integrated). `AutoARIMA` searches for the best orders. We use the fast `statsforecast` version.

In [ ]:
from sktime.forecasting.statsforecast import StatsForecastAutoARIMA

arima = StatsForecastAutoARIMA(sp=7)
arima.fit(y_train)
show(ARIMA=arima.predict(fh))

ETS and ARIMA see only the weekly season (`sp=7`). They have no idea that Halloween comes every year, so they cannot forecast the peak.

**TBATS** handles several and long seasonalities (it models them with sines and cosines). Fitting it is slower.

In [ ]:
from sktime.forecasting.tbats import TBATS

tbats = TBATS(sp=[7, 365.25], use_box_cox=False, use_trend=False,
              use_damped_trend=False, use_arma_errors=False, n_jobs=1)
tbats.fit(y_train)
show(TBATS=tbats.predict(fh))

**Prophet** (Meta, 2017) is a regression on time: a piecewise linear trend (it may bend at "changepoints"), a weekly and a yearly pattern (sines and cosines again) and holiday effects. It was built for business data with holidays, it is robust to missing days and outliers, and it is easy to tune by hand.

In [ ]:
from sktime.forecasting.fbprophet import Prophet

prophet = Prophet(yearly_seasonality=True, n_changepoints=10,
                  add_country_holidays={"country_name": "US"})
prophet.fit(y_train)
show(Prophet=prophet.predict(fh))

**Try it:** Prophet has a [list of settings](https://facebook.github.io/prophet/docs/quick_start.html). Change `seasonality_mode`, `changepoint_prior_scale` or the country. Can you beat MSTL?

---
## 7. Machine learning: a time series is a table

We can turn forecasting into ordinary regression: the input is a **window** of the last few days, the target is the next day. Each position of the window gives one row of a table:

In [ ]:
window = 7
X = pd.DataFrame({f"t-{window - i}": y_train.shift(window - i) for i in range(window)})
X["target (t)"] = y_train
X.dropna().round(2).tail()

**Look inside.** Every row is a training example: 7 days in, 1 day out. Now any regressor from scikit-learn works, e.g. LightGBM (gradient boosted trees, more in the next lesson).

To forecast 64 days, the model predicts tomorrow, appends the prediction to the window, predicts the day after, and so on (`strategy="recursive"`). sktime calls this **reduction**: forecasting is reduced to regression.

In [ ]:
from sktime.forecasting.compose import make_reduction
from lightgbm import LGBMRegressor

# n_jobs=1 avoids a crash on Macs, where LightGBM and PyTorch clash; on Colab it does not matter
lgbm = make_reduction(LGBMRegressor(verbose=-1, n_jobs=1), window_length=28, strategy="recursive")
lgbm.fit(y_train)
show(LightGBM=lgbm.predict(fh))

With a 28-day window, the model cannot see last October, so it cannot know about Halloween.

**Try it:** set `window_length=371` (a year and a week). Does it find the peak? What happens on a page without a yearly pattern?

---
## 8. Foundation models

Lesson 04: a transformer predicts the next token. What if the "tokens" are numbers from millions of time series? Since 2024, several companies have trained such **time series foundation models**. They work **zero-shot**: no training on our data, `fit` only stores the history (the *context*), and `predict` runs the pretrained transformer.

How do numbers become tokens? Roughly: scale the series (divide by its mean), cut it into **patches** of e.g. 16 days, and turn each patch into a vector, just like a word embedding. The model then predicts the next patches.

We try three open models (Apache 2.0 licence) through sktime, and Timestral, one of the best models in November 2026, directly:

| model | by | released | size |
|---|---|---|---|
| TimesFM 2.5 | Google | Sep 2025 | 231M |
| Chronos-2 | Amazon | Oct 2025 | 119M |
| TiRex-2 | NX-AI | Jun 2026 | 83M |
| Timestral | Mistral | Oct 2026 | 209M |

For comparison, GPT-2 from Lesson 04 had 124M parameters.

*(The first run downloads the weights. On a CPU, each model takes a minute or two.)*

In [ ]:
from sktime.forecasting.timesfm2 import TimesFM2Forecaster

timesfm = TimesFM2Forecaster()   # google/timesfm-2.5-200m-transformers
timesfm.fit(y_train)
show(TimesFM=timesfm.predict(fh))

In [ ]:
from sktime.forecasting.chronos2 import Chronos2Forecaster

chronos = Chronos2Forecaster("amazon/chronos-2")
chronos.fit(y_train)
show(Chronos2=chronos.predict(fh))

In [ ]:
from sktime.forecasting.tirex2 import TiRex2Forecaster

tirex = TiRex2Forecaster()   # NX-AI/TiRex-2
tirex.fit(y_train)
show(TiRex2=tirex.predict(fh))

**Timestral** is not in sktime yet, so we call it directly. It takes a plain array (pages × days) and returns quantiles for the next days. We wrap it in a small function that returns the same thing as sktime's `predict`.

*(Licence: free, except for companies with more than $20M revenue a month.)*

In [ ]:
import torch
from timestral import TimestralForecaster

timestral = TimestralForecaster.from_pretrained("mistralai/Timestral-209M",
                                                device="cuda" if torch.cuda.is_available() else "cpu")

def timestral_predict(y):
    """Forecast our 62 test days from a Series (one page) or a DataFrame (one column per page)."""
    df = y.to_frame() if isinstance(y, pd.Series) else y
    forecast = timestral.forecast(df.T.values, prediction_length=64)   # 64 days after the cutoff
    median = forecast.median[:, 0, 2:].cpu().numpy().T                 # skip the 2-day gap
    pred = pd.DataFrame(median, index=test_log.index, columns=df.columns)
    return pred.iloc[:, 0] if isinstance(y, pd.Series) else pred

show(Timestral=timestral_predict(y_train))

**Look inside.** What does Timestral return?

In [ ]:
forecast = timestral.forecast(y_train.values, prediction_length=64)
print("shape (series, variables, days, quantiles):", tuple(forecast.quantiles.shape))
print("quantile levels:", forecast.quantile_levels)

**Look inside.** How big is TimesFM? (sktime keeps the underlying PyTorch model in an attribute.)

In [ ]:
torch_models = [v for v in vars(timesfm).values() if hasattr(v, "parameters")]
print(type(torch_models[0]).__name__, f"{sum(p.numel() for p in torch_models[0].parameters()) / 1e6:.0f}M parameters")

**Uncertainty.** A single line is a lie: the future is uncertain. TimesFM, TiRex-2 and Timestral forecast **quantiles** (Chronos-2 too, but not through sktime). An 80 % prediction interval goes from the 10 % to the 90 % quantile:

In [ ]:
interval = timesfm.predict_interval(fh, coverage=0.8)
plot_series(y_train["2016-09":], y_test, timesfm.predict(fh),
            labels=["train", "test", "TimesFM"], pred_interval=interval);

**Try it:** how many test days fall outside the 80 % interval? (About 20 % would be honest.)

---
## 9. An ensemble

Different models make different mistakes. Averaging them often beats each one of them. At Google, the production forecasts were averages of many models. sktime has `EnsembleForecaster` for this, but since we already have the forecasts, we can simply average them:

In [ ]:
ensemble = (mstl.predict(fh) + chronos.predict(fh) + last_year) / 3
show(ensemble=ensemble)

The leaderboard for our page:

In [ ]:
pd.Series(leaderboard, name="sMAPE").sort_values().round(1)

**Try it:** go back to section 1, pick another page (e.g. `Main_Page_en.wikipedia.org_all-access_all-agents` or a random one) and run everything again. Is the order of the models the same?

---
## 10. All pages at once

One page is an anecdote. sktime can fit a model to every column of a DataFrame: it clones the forecaster and fits one copy per page.

In [ ]:
forecaster = NaiveForecaster(strategy="last", sp=7)
forecaster.fit(train_log)
forecaster.forecasters_.iloc[0, :3]   # look inside: one fitted forecaster per page

In [ ]:
def smape_all(y_pred):
    """Average sMAPE over all pages."""
    return np.mean([smape(test_log[c], y_pred[c]) for c in test_log.columns])

models = {
    "mean": NaiveForecaster(strategy="mean"),
    "last": NaiveForecaster(strategy="last"),
    "last week": NaiveForecaster(strategy="last", sp=7),
    "last 8 weeks": NaiveForecaster(strategy="mean", sp=7, window_length=56),
    "last year": NaiveForecaster(strategy="last", sp=364),
    "MSTL": StatsForecastMSTL(season_length=[7, 365]),
    "ETS": AutoETS(auto=True, sp=7),
    "ARIMA": StatsForecastAutoARIMA(sp=7),
    "Prophet": Prophet(yearly_seasonality=True, n_changepoints=10),
    "LightGBM": make_reduction(LGBMRegressor(verbose=-1, n_jobs=1), window_length=28),
}

all_pages = {}
all_preds = {}
for name, model in models.items():
    all_preds[name] = model.fit(train_log).predict(fh)
    all_pages[name] = smape_all(all_preds[name])
    print(f"{name:15s} {all_pages[name]:6.1f}")

And the foundation models. TimesFM and Timestral forecast each page separately, like the models above. Chronos-2 and TiRex-2 are different: they are **multivariate**, so they read the whole DataFrame as one series with many variables (one per page) and can use patterns shared between the pages.

In [ ]:
foundation = {"TimesFM": TimesFM2Forecaster(), "Chronos2": Chronos2Forecaster("amazon/chronos-2"), "TiRex2": TiRex2Forecaster()}
for name, model in foundation.items():
    all_preds[name] = model.fit(train_log).predict(fh)
    all_pages[name] = smape_all(all_preds[name])
    print(f"{name:15s} {all_pages[name]:6.1f}")

all_preds["Timestral"] = timestral_predict(train_log)
all_pages["Timestral"] = smape_all(all_preds["Timestral"])
print(f"{'Timestral':15s} {all_pages['Timestral']:6.1f}")

In [ ]:
pd.Series(all_pages, name=f"mean sMAPE, {train.shape[1]} pages").sort_values().round(1)

For comparison: the winner of the Kaggle competition had a sMAPE of about 35.5, on all 145k pages (including the hard, small ones).

**Try it:**
- Which pages are the hardest? Compute the sMAPE per page for the best model and look at the worst page.
- Does Chronos-2 gain from seeing all pages together? Run it in a loop, one page at a time, and compare.

---
## 11. Backtesting: is one test window enough?

We evaluated on a single 62-day window. Maybe we were lucky (or unlucky). **Backtesting** repeats the experiment: pretend it is an earlier date, forecast, compare, move forward, repeat. sktime does it with a *splitter* and `evaluate`:

In [ ]:
from sktime.split import ExpandingWindowSplitter
from sktime.forecasting.model_evaluation import evaluate

cv = ExpandingWindowSplitter(initial_window=365, step_length=56, fh=np.arange(3, 65))
print("number of backtest windows:", cv.get_n_splits(y_train))

results = evaluate(forecaster=NaiveForecaster(strategy="last", sp=7), y=y_train, cv=cv,
                   scoring=MeanAbsolutePercentageError(symmetric=True), return_data=True)
results[["cutoff", "test_MeanAbsolutePercentageError"]]

**Look inside.** Each row is one experiment: train on everything up to `cutoff`, forecast days 3–64 after it. The error changes a lot from window to window. *(Careful: here the error is computed on logarithms, so it is much smaller than our sMAPE on visits.)*

**Try it:** run the same backtest with `StatsForecastMSTL(season_length=[7, 365])`. Which model wins in more windows?

---
### Summary

- A time series model is just `fit` + `predict`; sktime gives one interface to dozens of them.
- Always start with naive baselines (last week, last year). They are fast and surprisingly hard to beat.
- Decomposition (trend + seasons) and classical models are fast and explainable.
- Machine learning needs the time series turned into a table (reduction).
- Foundation models forecast zero-shot, and are often among the best, but some are slow on a CPU and all of them miss what is not in the history.
- Evaluate on more than one window, and on more than one series.